In [1]:
import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

load_dotenv(override=True)

True

In [7]:
# MODEL = "gpt-6-luna"
MODEL = "gpt-5.4-mini"

model = OpenAIModel(client_args={"api_key": os.environ["OPENAI_API_KEY"]}, model_id=MODEL,)

agent = Agent(
    model=model,
    system_prompt="You are a concise, friendly assistant. Reply in a single short sentence.",
)

In [8]:
result = await agent.invoke_async("Say hello in Chinese.")

你好！

In [3]:
import board

board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.',
  'status': 'pending',
  'result': ''}]

In [12]:
board.show_board()

Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.

In [4]:
@tool
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

@tool
def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board.

    Args:
        goal_id: The id of the goal to break down.
        steps: Short descriptions of the steps to take, in order.
    """
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

@tool
def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) done and record a short result summary.

    Args:
        task_id: The id of the todo to mark done.
        result: A short summary of what was accomplished.
    """
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [9]:
board_agent = Agent(
    model=model,
    system_prompt="You help manage a shared todo board.",
    tools=[show_todos, complete_task],
)

In [10]:
result = await board_agent.invoke_async("What is on the board right now, and what is its status?")


Tool #1: show_todos
Here’s what’s on the board right now:

- **Task 1:** Read `notes.txt`, translate its contents into natural Spanish, and write the Spanish to `spanish.txt`.
  - **Status:** pending

There are no completed tasks yet.

In [11]:
workspace = Path("workspace").resolve()   # the only folder the agent may touch

filesystem = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
            cwd=str(workspace),  # start the server in the workspace so relative file names resolve there
        ),
        errlog=subprocess.DEVNULL,
    ),
    startup_timeout=60
)

In [13]:
file_agent = Agent(
    model=model,
    system_prompt="You can read and write files in your workspace. Use your tools to do what is asked.",
    tools=[filesystem],
)

In [14]:
result = await file_agent.invoke_async("Read notes.txt and summarize it in one short sentence.")


Tool #1: read_text_file
It says the team is building a small language tutor one task at a time, and each helper should carefully complete their assigned task.

In [15]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = Agent(
    model=model,
    system_prompt=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem],
)

board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.claim_todo(goal_id)

await worker.invoke_async("Please work the pending goal on the board.")
board.show_board()


Tool #1: show_todos

Tool #2: plan_steps

Tool #3: read_text_file

Tool #4: list_directory

Tool #5: write_file

Tool #6: complete_task

Tool #7: complete_task

Tool #8: complete_task

Tool #9: complete_task
Done.

Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.  Completed the translation workflow and created spanish.txt with the Spanish version of notes.txt.
  Step #2: Read notes.txt and extract the source text.  Read notes.txt and captured the source English text.
  Step #3: Translate the text into natural Spanish.  Translated the notes into natural Spanish.
  Step #4: Write the Spanish translation to spanish.txt.  Wrote the Spanish translation to spanish.txt.

In [16]:
import os
from pathlib import Path

from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport

load_dotenv(override=True)

True

In [24]:
MODEL = "openai-chat:gpt-5.4-mini"
# MODEL = "openai-chat:gpt-6-luna"

agent = Agent(
    MODEL,
    instructions="You are a concise, friendly assistant. Reply in a single short sentence.",
)

In [18]:
result = await agent.run("Say hello in Spanish.")
print(result.output)

¡Hola!


In [19]:
import board

board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.',
  'status': 'pending',
  'result': ''}]

In [20]:
board.show_board()

Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.

In [21]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [25]:
board_agent = Agent(
    MODEL,
    instructions="You help manage a shared todo board.",
    tools=[show_todos, complete_task],
)

In [26]:
result = await board_agent.run("What is on the board right now, and what is its status?")
print(result.output)

There is 1 item on the board:

- **Task 1:** “Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.”
  - **Status:** pending


In [27]:
workspace = Path("workspace").resolve()   # the only folder the agent may touch

filesystem = MCPToolset(
    StdioTransport(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
        cwd=str(workspace),  # start the server in the workspace so relative file names resolve there
        log_file=Path(os.devnull),
    ), 
    init_timeout=60
)

In [28]:
file_agent = Agent(
    MODEL,
    instructions="You can read and write files in your workspace. Use your tools to do what is asked.",
    toolsets=[filesystem],
)

In [29]:
async with file_agent:
    result = await file_agent.run("Read notes.txt and summarize it in one short sentence.")
print(result.output)

It says the team is building a small language tutor one task at a time, with each helper carefully completing one assigned piece.


In [30]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = Agent(
    MODEL,
    instructions=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task],
    toolsets=[filesystem],
)

board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.claim_todo(goal_id)

async with worker:
    result = await worker.run("Please work the pending goal on the board.")
print(result.output)
board.show_board()

Done — I translated `notes.txt` into Spanish and wrote it to `spanish.txt`.


Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.  Completed translation workflow and created spanish.txt.
  Step #2: Inspect notes.txt and understand the source text.  Read notes.txt and understood the source text.
  Step #3: Translate the text into natural Spanish.  Translated the notes into natural Spanish.
  Step #4: Write the translation to spanish.txt.  Wrote the Spanish translation to spanish.txt.
  Step #5: Verify the output and mark the goal complete.  Verified the translation file was created successfully.